# Multi-Head Attention
:label:`sec_multihead-attention`


In practice, given the same set of queries, keys, and values we may want our model to combine knowledge from
different behaviors of the same attention mechanism,
such as capturing dependencies of various ranges
(e.g., shorter-range vs. longer-range) within a sequence.
Thus, it may be beneficial to allow our attention mechanism to jointly use different representation subspaces of queries, keys, and values.


To this end, instead of performing 
a single attention pooling,
queries, keys, and values
can be transformed
with $h$ independently learned linear projections.
Then these $h$ projected queries, keys, and values
are fed into attention pooling in parallel.
In the end,
$h$ attention-pooling outputs
are concatenated and 
transformed with another learned linear projection
to produce the final output.
This design
is called *multi-head attention*,
where each of the $h$ attention pooling outputs
is a *head* :cite:`Vaswani.Shazeer.Parmar.ea.2017`.
Using fully connected layers
to perform learnable linear transformations,
:numref:`fig_multi-head-attention`
describes multi-head attention.

![Multi-head attention, where multiple heads are concatenated then linearly transformed.](../img/multi-head-attention.svg)
:label:`fig_multi-head-attention`


In [1]:
import math
import torch
from torch import nn
from d2l import torch as d2l

## Model

Before providing the implementation of multi-head attention,
let's formalize this model mathematically.
Given a query $\mathbf{q} \in \mathbb{R}^{d_q}$,
a key $\mathbf{k} \in \mathbb{R}^{d_k}$,
and a value $\mathbf{v} \in \mathbb{R}^{d_v}$,
each attention head $\mathbf{h}_i$  ($i = 1, \ldots, h$)
is computed as

$$\mathbf{h}_i = f(\mathbf W_i^{(q)}\mathbf q, \mathbf W_i^{(k)}\mathbf k,\mathbf W_i^{(v)}\mathbf v) \in \mathbb R^{p_v},$$

where 
$\mathbf W_i^{(q)}\in\mathbb R^{p_q\times d_q}$,
$\mathbf W_i^{(k)}\in\mathbb R^{p_k\times d_k}$,
and $\mathbf W_i^{(v)}\in\mathbb R^{p_v\times d_v}$
are learnable parameters and
$f$ is attention pooling,
such as
additive attention and scaled dot product attention
in :numref:`sec_attention-scoring-functions`.
The multi-head attention output
is another linear transformation via 
learnable parameters
$\mathbf W_o\in\mathbb R^{p_o\times h p_v}$
of the concatenation of $h$ heads:

$$\mathbf W_o \begin{bmatrix}\mathbf h_1\\\vdots\\\mathbf h_h\end{bmatrix} \in \mathbb{R}^{p_o}.$$

Based on this design, each head may attend
to different parts of the input.
More sophisticated functions 
than the simple weighted average can be expressed.

## Implementation

In our implementation,
we [**choose the scaled dot product attention
for each head**] of the multi-head attention.
To avoid significant growth of computational cost and parametrization cost,
we set $p_q = p_k = p_v = p_o / h$.
Note that $h$ heads can be computed in parallel
if we set the number of outputs 
of linear transformations
for the query, key, and value
to $p_q h = p_k h = p_v h = p_o$.
In the following implementation,
$p_o$ is specified via the argument `num_hiddens`.


In [2]:
class MultiHeadAttention(d2l.Module):  #@save
    """Multi-head attention."""
    def __init__(self, num_hiddens, num_heads, dropout, bias=False, **kwargs):
        super().__init__()
        self.num_heads = num_heads
        self.attention = d2l.DotProductAttention(dropout)
        self.W_q = nn.LazyLinear(num_hiddens, bias=bias)
        self.W_k = nn.LazyLinear(num_hiddens, bias=bias)
        self.W_v = nn.LazyLinear(num_hiddens, bias=bias)
        self.W_o = nn.LazyLinear(num_hiddens, bias=bias)

    def forward(self, queries, keys, values, valid_lens):
        # Shape of queries, keys, or values:
        # (batch_size, no. of queries or key-value pairs, num_hiddens)
        # Shape of valid_lens: (batch_size,) or (batch_size, no. of queries)
        # After transposing, shape of output queries, keys, or values:
        # (batch_size * num_heads, no. of queries or key-value pairs,
        # num_hiddens / num_heads)
        queries = self.transpose_qkv(self.W_q(queries))
        keys = self.transpose_qkv(self.W_k(keys))
        values = self.transpose_qkv(self.W_v(values))

        if valid_lens is not None:
            # On axis 0, copy the first item (scalar or vector) for num_heads
            # times, then copy the next item, and so on
            valid_lens = torch.repeat_interleave(
                valid_lens, repeats=self.num_heads, dim=0)

        # Shape of output: (batch_size * num_heads, no. of queries,
        # num_hiddens / num_heads)
        output = self.attention(queries, keys, values, valid_lens)
        # Shape of output_concat: (batch_size, no. of queries, num_hiddens)
        output_concat = self.transpose_output(output)
        return self.W_o(output_concat)

To allow for [**parallel computation of multiple heads**],
the above `MultiHeadAttention` class uses two transposition methods as defined below.
Specifically,
the `transpose_output` method reverses the operation
of the `transpose_qkv` method.


In [3]:
@d2l.add_to_class(MultiHeadAttention)  #@save
def transpose_qkv(self, X):
    """Transposition for parallel computation of multiple attention heads."""
    # Shape of input X: (batch_size, no. of queries or key-value pairs,
    # num_hiddens). Shape of output X: (batch_size, no. of queries or
    # key-value pairs, num_heads, num_hiddens / num_heads)
    X = X.reshape(X.shape[0], X.shape[1], self.num_heads, -1)
    # Shape of output X: (batch_size, num_heads, no. of queries or key-value
    # pairs, num_hiddens / num_heads)
    X = X.permute(0, 2, 1, 3)
    # Shape of output: (batch_size * num_heads, no. of queries or key-value
    # pairs, num_hiddens / num_heads)
    return X.reshape(-1, X.shape[2], X.shape[3])

@d2l.add_to_class(MultiHeadAttention)  #@save
def transpose_output(self, X):
    """Reverse the operation of transpose_qkv."""
    X = X.reshape(-1, self.num_heads, X.shape[1], X.shape[2])
    X = X.permute(0, 2, 1, 3)
    return X.reshape(X.shape[0], X.shape[1], -1)

Let's [**test our implemented**] `MultiHeadAttention` class
using a toy example where keys and values are the same.
As a result,
the shape of the multi-head attention output
is (`batch_size`, `num_queries`, `num_hiddens`).


In [4]:
num_hiddens, num_heads = 100, 5
attention = MultiHeadAttention(num_hiddens, num_heads, 0.5)
batch_size, num_queries, num_kvpairs = 2, 4, 6
valid_lens = torch.tensor([3, 2])
X = torch.ones((batch_size, num_queries, num_hiddens))
Y = torch.ones((batch_size, num_kvpairs, num_hiddens))
d2l.check_shape(attention(X, Y, Y, valid_lens),
                (batch_size, num_queries, num_hiddens))

## Summary

Multi-head attention combines knowledge of the same attention pooling 
via different representation subspaces of queries, keys, and values.
To compute multiple heads of multi-head attention in parallel, 
proper tensor manipulation is needed.


## Exercises

1. Visualize attention weights of multiple heads in this experiment.
1. Suppose that we have a trained model based on multi-head attention and we want to prune less important attention heads to increase the prediction speed. How can we design experiments to measure the importance of an attention head?


[Discussions](https://discuss.d2l.ai/t/1635)


I'll help you with visualizing attention weights and designing experiments for pruning attention heads in transformer models.

```markdown
# Attention Visualization and Head Importance Evaluation

## Question 1: Visualizing Multi-Head Attention Weights

To visualize attention weights from multiple heads in a transformer model, we need to:
1. Extract the attention weights after a forward pass
2. Process them into a suitable format for visualization
3. Create meaningful visualizations that highlight patterns

Here's a complete implementation:

```python
import torch
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from transformers import AutoTokenizer, AutoModel
import matplotlib.ticker as ticker
from matplotlib.colors import LinearSegmentedColormap

class AttentionVisualizer:
    def __init__(self, model_name="gpt2-medium"):
        """Initialize with a pretrained model."""
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name, output_attentions=True)
        self.model.eval()
        
    def get_attention_weights(self, text):
        """Extract attention weights for the given text."""
        # Tokenize input
        inputs = self.tokenizer(text, return_tensors="pt")
        
        # Run model with attention outputs
        with torch.no_grad():
            outputs = self.model(**inputs, output_attentions=True)
        
        # Get attention weights
        # Shape: [num_layers, batch_size, num_heads, seq_len, seq_len]
        attention_weights = outputs.attentions
        
        # Convert to numpy for easier visualization
        attention_weights = [layer.cpu().numpy() for layer in attention_weights]
        
        # Get tokens for better visualization
        tokens = self.tokenizer.convert_ids_to_tokens(inputs.input_ids[0])
        
        return attention_weights, tokens
    
    def plot_attention_heads(self, text, layer=0, heads=None, figsize=(16, 8)):
        """
        Visualize specific attention heads in a given layer.
        
        Args:
            text: Input text to visualize attention for
            layer: Layer index to visualize
            heads: List of head indices to visualize, or None for all heads
            figsize: Size of the figure
        """
        # Get attention weights and tokens
        attention_weights, tokens = self.get_attention_weights(text)
        
        # Get weights for specific layer
        layer_weights = attention_weights[layer][0]  # batch size = 1
        
        # Determine heads to visualize
        num_heads = layer_weights.shape[0]
        if heads is None:
            heads = range(num_heads)
        else:
            heads = [h for h in heads if h < num_heads]
        
        # Create figure
        n_heads = len(heads)
        fig, axes = plt.subplots(1, n_heads, figsize=figsize)
        if n_heads == 1:
            axes = [axes]
        
        # Custom colormap: white to blue gradient
        colors = [(1, 1, 1), (0, 0.4, 0.8)]  # Light blue to dark blue
        attention_cmap = LinearSegmentedColormap.from_list("attention_cmap", colors)
        
        # Plot each attention head
        for i, head_idx in enumerate(heads):
            ax = axes[i]
            
            # Get attention weights for this head
            attn = layer_weights[head_idx]
            
            # Create heatmap
            sns.heatmap(
                attn,
                xticklabels=tokens,
                yticklabels=tokens,
                cmap=attention_cmap,
                vmin=0,
                vmax=1,
                ax=ax,
                square=True,
                cbar=i == n_heads-1  # Only show colorbar for last plot
            )
            
            # Configure ticks
            ax.set_xticklabels(tokens, rotation=90)
            ax.set_yticklabels(tokens, rotation=0)
            
            # Reduce tick frequency for long sequences
            if len(tokens) > 20:
                ax.xaxis.set_major_locator(ticker.MultipleLocator(5))
                ax.yaxis.set_major_locator(ticker.MultipleLocator(5))
            
            ax.set_title(f"Head {head_idx}")
            
            # Add labels
            if i == 0:
                ax.set_ylabel("Query tokens")
            if i == n_heads // 2:
                ax.set_xlabel("Key tokens")
        
        plt.tight_layout()
        plt.suptitle(f"Attention patterns in layer {layer}", y=1.05)
        return fig
    
    def plot_attention_across_layers(self, text, head=0, figsize=(16, 12)):
        """
        Visualize attention patterns of a specific head across all layers.
        
        Args:
            text: Input text to visualize attention for
            head: Head index to visualize
            figsize: Size of the figure
        """
        # Get attention weights and tokens
        attention_weights, tokens = self.get_attention_weights(text)
        
        # Determine number of layers
        n_layers = len(attention_weights)
        
        # Create figure with subplots arranged in a grid
        n_cols = min(4, n_layers)
        n_rows = (n_layers + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
        if n_rows == 1 and n_cols == 1:
            axes = np.array([axes])
        axes = axes.flatten()
        
        # Custom colormap
        attention_cmap = sns.color_palette("YlOrRd", as_cmap=True)
        
        # Plot each layer
        for layer_idx in range(n_layers):
            ax = axes[layer_idx]
            
            # Get attention weights for this layer and head
            attn = attention_weights[layer_idx][0, head]
            
            # Create heatmap
            sns.heatmap(
                attn,
                xticklabels=tokens if layer_idx >= (n_rows-1)*n_cols else [],
                yticklabels=tokens if layer_idx % n_cols == 0 else [],
                cmap=attention_cmap,
                vmin=0,
                vmax=np.max(attn),  # Scale to max value in this head
                ax=ax,
                square=True,
                cbar=False
            )
            
            # Configure ticks
            if layer_idx >= (n_rows-1)*n_cols:
                ax.set_xticklabels(tokens, rotation=90, fontsize=8)
            if layer_idx % n_cols == 0:
                ax.set_yticklabels(tokens, rotation=0, fontsize=8)
            
            # Reduce tick frequency for long sequences
            if len(tokens) > 20:
                ax.xaxis.set_major_locator(ticker.MultipleLocator(5))
                ax.yaxis.set_major_locator(ticker.MultipleLocator(5))
            
            ax.set_title(f"Layer {layer_idx}")
        
        # Add colorbar
        cbar_ax = fig.add_axes([0.92, 0.3, 0.02, 0.4])
        sm = plt.cm.ScalarMappable(cmap=attention_cmap)
        sm.set_array([])
        plt.colorbar(sm, cax=cbar_ax)
        
        # Hide empty subplots
        for i in range(n_layers, len(axes)):
            axes[i].axis('off')
        
        plt.tight_layout()
        plt.suptitle(f"Attention patterns of head {head} across layers", y=0.95)
        return fig
    
    def plot_attention_matrix(self, text, layer=0, head=0):
        """Plot a single attention matrix with improved visual representation."""
        attention_weights, tokens = self.get_attention_weights(text)
        attn = attention_weights[layer][0, head]
        
        # Create figure
        fig, ax = plt.subplots(figsize=(12, 10))
        
        # Custom colormap with white-to-deep blue gradient
        cmap = LinearSegmentedColormap.from_list(
            "custom_cmap", 
            [(1, 1, 1), (0.9, 0.9, 1), (0.6, 0.6, 0.9), (0.2, 0.2, 0.7), (0, 0, 0.5)]
        )
        
        # Create heatmap with grid lines
        sns.heatmap(
            attn,
            xticklabels=tokens,
            yticklabels=tokens,
            cmap=cmap,
            vmin=0,
            vmax=max(0.001, attn.max()),  # Dynamic scale with minimum threshold
            ax=ax,
            square=True,
            linewidths=0.1,
            linecolor='#cccccc',
            cbar_kws={"label": "Attention weight"}
        )
        
        # Add labels and title
        ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=10)
        ax.set_yticklabels(tokens, rotation=0, fontsize=10)
        ax.set_xlabel("Key tokens", fontsize=12)
        ax.set_ylabel("Query tokens", fontsize=12)
        ax.set_title(f"Attention pattern (Layer {layer}, Head {head})", fontsize=14)
        
        # Add grid at token boundaries
        ax.grid(False)
        
        # Highlight diagonal (self-attention)
        n = len(tokens)
        for i in range(n):
            ax.add_patch(plt.Rectangle((i, i), 1, 1, fill=False, edgecolor='black', lw=0.7))
        
        # Add text annotations for high attention values
        for i in range(n):
            for j in range(n):
                if attn[i, j] > 0.2:  # Only label strong connections
                    ax.text(j + 0.5, i + 0.5, f'{attn[i, j]:.2f}',
                           ha="center", va="center", color="black" if attn[i, j] < 0.5 else "white",
                           fontsize=8)
        
        plt.tight_layout()
        return fig
    
    def visualize_token_relationships(self, text, layer=11, top_k=3):
        """
        Visualize which tokens each token is attending to the most.
        Useful for understanding semantic relationships.
        """
        attention_weights, tokens = self.get_attention_weights(text)
        
        # Average attention across all heads in the specified layer
        avg_attn = attention_weights[layer][0].mean(axis=0)
        
        fig, ax = plt.subplots(figsize=(14, len(tokens) * 0.5 + 2))
        
        # For each token, find the tokens it pays most attention to
        relationships = []
        for i, token in enumerate(tokens):
            # Get attention weights for this token
            attn_weights = avg_attn[i]
            
            # Find top-k attended tokens
            top_indices = np.argsort(attn_weights)[-top_k:][::-1]
            top_tokens = [tokens[idx] for idx in top_indices]
            top_weights = [attn_weights[idx] for idx in top_indices]
            
            # Add to relationships list
            relationships.append((token, list(zip(top_tokens, top_weights))))
        
        # Create visual representation
        y_positions = np.arange(len(tokens))
        
        # Plot tokens
        ax.barh(y_positions, [0.3] * len(tokens), left=0, height=0.5, color='lightgray', alpha=0.3)
        for i, token in enumerate(tokens):
            ax.text(0.15, i, token, ha='center', va='center', fontsize=10)
        
        # Plot relationships
        for i, (token, top_relations) in enumerate(relationships):
            for j, (related_token, weight) in enumerate(top_relations):
                # Find position of related token
                related_idx = tokens.index(related_token)
                
                # Draw arrow with thickness proportional to attention weight
                arrow_width = weight * 3
                arrow_style = f'simple, head_width={arrow_width*2+0.05}, head_length=0.15'
                
                # Different colors for different top-k ranks
                colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
                
                # Draw curved arrow
                ax.annotate('', 
                           xy=(0.5 + j*0.2, related_idx), 
                           xytext=(0.5 + j*0.2, i),
                           arrowprops=dict(arrowstyle=arrow_style, 
                                          color=colors[j % len(colors)],
                                          alpha=min(1.0, weight*1.5),
                                          connectionstyle='arc3,rad=0.2'))
                
                # Add weight label
                midpoint_y = (i + related_idx) / 2
                ax.text(0.5 + j*0.2 + 0.05, midpoint_y, f'{weight:.2f}', 
                       ha='left', va='center', fontsize=8, color=colors[j % len(colors)])
        
        # Configure plot
        ax.set_xlim(0, 1.2)
        ax.set_ylim(-1, len(tokens))
        ax.invert_yaxis()  # To match token order in text
        ax.axis('off')
        plt.tight_layout()
        plt.title(f"Top-{top_k} token relationships in layer {layer}", fontsize=14)
        
        return fig

# Example usage
def run_attention_visualization_experiment():
    # Create visualizer
    visualizer = AttentionVisualizer("gpt2-medium")
    
    # Sample text
    text = "The transformer model revolutionized natural language processing."
    
    # Visualize multiple heads in a single layer
    fig1 = visualizer.plot_attention_heads(text, layer=5, heads=[0, 3, 7, 11])
    
    # Visualize a single head across layers
    fig2 = visualizer.plot_attention_across_layers(text, head=7)
    
    # Detailed visualization of a single attention matrix
    fig3 = visualizer.plot_attention_matrix(text, layer=11, head=5)
    
    # Visualize token relationships
    fig4 = visualizer.visualize_token_relationships(text, layer=11, top_k=3)
    
    # Save figures
    fig1.savefig("multi_head_attention.png", bbox_inches="tight", dpi=300)
    fig2.savefig("cross_layer_attention.png", bbox_inches="tight", dpi=300)
    fig3.savefig("detailed_attention_matrix.png", bbox_inches="tight", dpi=300)
    fig4.savefig("token_relationships.png", bbox_inches="tight", dpi=300)
    
    return {
        "multi_head": fig1,
        "cross_layer": fig2,
        "detailed_matrix": fig3,
        "token_relationships": fig4
    }

# For interactive usage in a notebook, you can call:
# figs = run_attention_visualization_experiment()
# display(figs["multi_head"])
```

## Question 2: Measuring Importance of Attention Heads for Pruning

To design experiments for measuring the importance of attention heads and identifying which ones to prune, we can use several complementary approaches:

### 1. Head Ablation Study

The most direct approach is to measure the performance impact of removing individual heads:

```python
import torch
import torch.nn as nn
import numpy as np
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from datasets import load_dataset
from torch.utils.data import DataLoader
import tqdm

class HeadImportanceEvaluator:
    def __init__(self, model_name, dataset_name="glue", subset="sst2"):
        """Initialize with a model and dataset for evaluation."""
        self.model = AutoModelForSequenceClassification.from_pretrained(model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.dataset = load_dataset(dataset_name, subset)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        
        # Get model architecture details
        self.num_layers = self.model.config.num_hidden_layers
        self.num_heads = self.model.config.num_attention_heads
        
    def prepare_dataloader(self, split="validation", batch_size=32):
        """Prepare dataloader for evaluation."""
        dataset = self.dataset[split]
        
        def tokenize_function(examples):
            return self.tokenizer(
                examples["sentence"] if "sentence" in examples else examples["text"],
                padding="max_length",
                truncation=True,
                max_length=128
            )
        
        tokenized_dataset = dataset.map(tokenize_function, batched=True)
        tokenized_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
        
        return DataLoader(tokenized_dataset, batch_size=batch_size)
    
    def evaluate_baseline(self, dataloader):
        """Evaluate model without any pruning."""
        self.model.eval()
        correct = 0
        total = 0
        
        with torch.no_grad():
            for batch in tqdm.tqdm(dataloader):
                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                labels = batch["label"].to(self.device)
                
                outputs = self.model(input_ids, attention_mask=attention_mask)
                logits = outputs.logits
                
                _, predicted = torch.max(logits, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
        
        accuracy = correct / total
        return accuracy
    
    def ablate_head(self, layer_idx, head_idx):
        """
        Create a mask to ablate a specific attention head during forward pass.
        This is a non-destructive operation that doesn't modify the model weights.
        """
        # Get attention modules (this depends on model architecture)
        if hasattr(self.model, "transformer"):
            attn_modules = self.model.transformer.h
            attn_key = "attn"
        elif hasattr(self.model, "bert"):
            attn_modules = self.model.bert.encoder.layer
            attn_key = "attention.self"
        else:
            raise ValueError("Unsupported model architecture")
        
        # Register forward hook to zero out specific head
        def mask_head(module, input, output):
            # Output is typically (batch_size, num_heads, seq_len, head_dim)
            # Zero out the specific head
            output[0][:, head_idx] = 0.0
            return output
        
        # Find attention layer and register hook
        layer = attn_modules[layer_idx]
        if attn_key == "attn":
            # GPT-style models
            attn_module = layer.attn
        else:
            # BERT-style models
            attn_module = getattr(layer, attn_key)
        
        # Register hook on attention outputs
        handle = attn_module.register_forward_hook(mask_head)
        
        return handle
    
    def measure_head_importance_by_ablation(self, batch_size=32):
        """
        Measure importance of each head by ablating it and measuring performance drop.
        """
        # Evaluate baseline performance first
        dataloader = self.prepare_dataloader(batch_size=batch_size)
        baseline_accuracy = self.evaluate_baseline(dataloader)
        print(f"Baseline accuracy: {baseline_accuracy:.4f}")
        
        # Matrix to store importance scores
        importance_scores = np.zeros((self.num_layers, self.num_heads))
        
        # For each layer and head, measure performance after ablation
        for layer_idx in range(self.num_layers):
            for head_idx in range(self.num_heads):
                print(f"Evaluating Layer {layer_idx}, Head {head_idx}")
                
                # Ablate the head
                handle = self.ablate_head(layer_idx, head_idx)
                
                # Evaluate
                ablated_accuracy = self.evaluate_baseline(dataloader)
                
                # Compute performance drop as importance
                importance = baseline_accuracy - ablated_accuracy
                importance_scores[layer_idx, head_idx] = importance
                
                # Remove hook
                handle.remove()
                
                print(f"  Accuracy drop: {importance:.4f}")
                
        return importance_scores
    
    def measure_head_importance_by_gradient(self, batch_size=16, num_batches=10):
        """
        Measure importance of each head by computing gradient magnitudes.
        """
        self.model.train()  # Need gradients
        dataloader = self.prepare_dataloader(batch_size=batch_size)
        
        # Matrix to store importance scores
        importance_scores = np.zeros((self.num_layers, self.num_heads))
        
        # Extract attention modules (architecture-dependent)
        if hasattr(self.model, "transformer"):
            attn_modules = self.model.transformer.h
            weight_key = "attn.c_attn.weight"
        elif hasattr(self.model, "bert"):
            attn_modules = self.model.bert.encoder.layer
            qkv_keys = ["attention.self.query.weight", 
                       "attention.self.key.weight", 
                       "attention.self.value.weight"]
        else:
            raise ValueError("Unsupported model architecture")
        
        # Process batches and accumulate gradients
        batch_count = 0
        for batch in dataloader:
            if batch_count >= num_batches:
                break
                
            input_ids = batch["input_ids"].to(self.device)
            attention_mask = batch["attention_mask"].to(self.device)
            labels = batch["label"].to(self.device)
            
            # Forward pass
            outputs = self.model(input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            
            # Backward pass
            loss.backward()
            
            # Accumulate gradient magnitudes for each head
            for layer_idx in range(self.num_layers):
                layer = attn_modules[layer_idx]
                
                if hasattr(self.model, "transformer"):
                    # For GPT-style models with combined QKV weights
                    # Extract portions of the weight matrix corresponding to each head
                    weight = getattr(layer, weight_key.split('.')[0])
                    for param_name, param in weight.named_parameters():
                        if param_name == weight_key.split('.')[-1]:
                            grad = param.grad
                            if grad is not None:
                                head_size = self.model.config.hidden_size // self.num_heads
                                for head_idx in range(self.num_heads):
                                    # Calculate gradient magnitude for this head
                                    start_idx = head_idx * head_size
                                    end_idx = (head_idx + 1) * head_size
                                    head_grad = grad[:, start_idx:end_idx]
                                    importance_scores[layer_idx, head_idx] += torch.norm(head_grad).item()
                
                else:
                    # For BERT-style models with separate Q,K,V weights
                    for key in qkv_keys:
                        parts = key.split('.')
                        module = layer
                        for part in parts[:-1]:
                            module = getattr(module, part)
                        param = getattr(module, parts[-1])
                        grad = param.grad
                        
                        if grad is not None:
                            head_size = self.model.config.hidden_size // self.num_heads
                            for head_idx in range(self.num_heads):
                                start_idx = head_idx * head_size
                                end_idx = (head_idx + 1) * head_size
                                head_grad = grad[:, start_idx:end_idx]
                                importance_scores[layer_idx, head_idx] += torch.norm(head_grad).item() / len(qkv_keys)
            
            # Zero gradients before next batch
            self.model.zero_grad()
            batch_count += 1
        
        # Normalize by number of batches
        importance_scores /= batch_count
        
        return importance_scores
    
    def measure_head_importance_by_attention_distribution(self, batch_size=32, num_batches=10):
        """
        Measure head importance by analyzing attention distribution.
        Heads with more focused attention (lower entropy) might be more important.
        """
        self.model.eval()
        dataloader = self.prepare_dataloader(batch_size=batch_size)
        
        # Matrix to store importance scores
        importance_scores = np.zeros((self.num_layers, self.num_heads))
        
        # Lists to store attention distributions
        attention_entropies = [[[] for _ in range(self.num_heads)] for _ in range(self.num_layers)]
        attention_confidences = [[[] for _ in range(self.num_heads)] for _ in range(self.num_layers)]
        
        # Process batches
        batch_count = 0
        with torch.no_grad():
            for batch in dataloader:
                if batch_count >= num_batches:
                    break
                    
                input_ids = batch["input_ids"].to(self.device)
                attention_mask = batch["attention_mask"].to(self.device)
                
                # Forward pass with output_attentions=True
                outputs = self.model(
                    input_ids, 
                    attention_mask=attention_mask,
                    output_attentions=True
                )
                
                # Get attention weights
                # Shape: (batch_size, num_layers, num_heads, seq_len, seq_len)
                attentions = outputs.attentions
                
                # For each layer and head
                for layer_idx in range(self.num_layers):
                    layer_attentions = attentions[layer_idx]
                    
                    for head_idx in range(self.num_heads):
                        head_attentions = layer_attentions[:, head_idx]  # (batch_size, seq_len, seq_len)
                        
                        # Calculate metrics for each example in batch
                        for example_idx in range(head_attentions.size(0)):
                            attn = head_attentions[example_idx]
                            
                            # Apply attention mask to ignore padding
                            mask = attention_mask[example_idx].unsqueeze(-1).expand_as(attn)
                            attn = attn * mask
                            
                            # Calculate entropy of attention distribution
                            # Low entropy = more focused attention = potentially more important
                            for i in range(attn.size(0)):
                                if attention_mask[example_idx, i] == 1:  # Consider only non-padding tokens
                                    dist = attn[i, :attention_mask[example_idx].sum().int()]
                                    dist = dist / (dist.sum() + 1e-12)  # Normalize
                                    
                                    # Shannon entropy
                                    entropy = -torch.sum(dist * torch.log2(dist + 1e-12)).item()
                                    attention_entropies[layer_idx][head_idx].append(entropy)
                                    
                                    # Confidence (max attention weight)
                                    confidence = torch.max(dist).item()
                                    attention_confidences[layer_idx][head_idx].append(confidence)
                
                batch_count += 1
        
        # Calculate importance scores based on entropy and confidence
        for layer_idx in range(self.num_layers):
            for head_idx in range(self.num_heads):
                # Lower entropy = higher importance
                mean_entropy = np.mean(attention_entropies[layer_idx][head_idx])
                mean_confidence = np.mean(attention_confidences[layer_idx][head_idx])
                
                # Combined score (normalized between 0 and 1)
                importance_scores[layer_idx, head_idx] = mean_confidence * (1 - mean_entropy / np.log2(self.model.config.max_position_embeddings))
        
        # Min-max normalize
        importance_scores = (importance_scores - importance_scores.min()) / (importance_scores.max() - importance_scores.min() + 1e-12)
        
        return importance_scores
    
    def prune_heads(self, importance_scores, prune_percentage=0.3):
        """
        Determine which heads to prune based on importance scores.
        
        Args:
            importance_scores: Matrix of shape (num_layers, num_heads) with importance values
            prune_percentage: Percentage of heads to prune (0.0 to 1.0)
        
        Returns:
            Dictionary mapping layer indices to lists of heads to prune
        """
        # Flatten importance scores
        flat_importances = importance_scores.flatten()
        total_heads = len(flat_importances)
        
        # Determine number of heads to prune
        num_to_prune = int(total_heads * prune_percentage)
        
        # Get indices of least important heads
        flat_indices = np.argsort(flat_importances)[:num_to_prune]
        
        # Convert flat indices to (layer_idx, head_idx) pairs
        to_prune = {}
        for flat_idx in flat_indices:
            layer_idx = flat_idx // self.num_heads
            head_idx = flat_idx % self.num_heads
            
            if layer_idx not in to_prune:
                to_prune[layer_idx] = []
            to_prune[layer_idx].append(head_idx)
        
        return to_prune
    
    def apply_pruning(self, heads_to_prune):
        """
        Apply pruning to the model by actually removing the specified heads.
        
        This is model-dependent and may require modification based on architecture.
        For HuggingFace transformers, we can use the prune_heads method.
        """
        # For HuggingFace models
        if hasattr(self.model, "prune_heads"):
            self.model.prune_heads(heads_to_prune)
        else:
            # For models like BERT
            if hasattr(self.model, "bert"):
                self.model.bert.prune_heads(heads_to_prune)
            # For models like GPT
            elif hasattr(self.model, "transformer"):
                self.model.transformer.prune_heads(heads_to_prune)
            else:
                raise ValueError("Pruning not supported for this model")
        
        print(f"Pruned {sum(len(heads) for heads in heads_to_prune.values())} attention heads")
        
    def evaluate_progressive_pruning(self, method="ablation", prune_steps=10):
        """
        Evaluate model performance with progressively more heads pruned.
        
        Args:
            method: Method to compute importance ('ablation', 'gradient', or 'attention')
            prune_steps: Number of pruning steps to evaluate
        
        Returns:
            Dictionary with pruning percentages and corresponding accuracies
        """
        # Compute head importance
        if method == "ablation":
            importance_scores = self.measure_head_importance_by_ablation()
        elif method == "gradient":
            importance